# Efficiency plotting 

Copied most functionality from Moon's https://github.com/wjdanswjddl/cafpyana/blob/release/numucc_1p0pi/analysis_village/numucc_1p0pi/notebooks/event_selection.ipynb

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# print avaialbe memory
import psutil
print(psutil.virtual_memory())

svmem(total=1035267956736, available=918952869888, percent=11.2, used=116315086848, free=669639610368, active=103198879744, inactive=150724595712, buffers=7766016, cached=255913537536, shared=98742272, slab=108166696960)


In [3]:
import os
import pandas as pd
import numpy as np
import sys
from pathlib import Path
from os import makedirs, path
from datetime import datetime
import pickle

# local imports
# sys.path.append('../../../')
cwd = Path.cwd().resolve()
# repo_root = next((candidate for candidate in [cwd, *cwd.parents] if (candidate / 'analysis_village').exists()), None)
# if repo_root is None:
#     raise RuntimeError('Could not locate the repository root from the current notebook working directory')
repo_root = Path('/nashome/m/micarrig/sbnd/nueCCNp/cafpyana')
sys.path.append(str(repo_root))
from analysis_village.nueNp0Pi.config.plots import VariableConfig
from analysis_village.nueNp0Pi.config.settings import *
from analysis_village.nueNp0Pi.selections import *
from analysis_village.nueNp0Pi.utils import *
from pyanalib.split_df_helpers import *
from pyanalib.pandas_helpers import *
from pyanalib.covariance import *

import matplotlib.pyplot as plt 
from matplotlib.patches import Patch

plt.style.use(repo_root / 'analysis_village' / 'nueNp0Pi' / 'notebooks' / 'presentation.mplstyle')

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
# turn off RuntimeWarning
warnings.filterwarnings('ignore', category=RuntimeWarning)

os.environ['CAFPYANA_LOG_LEVEL'] = 'DEBUG'
import os

#TODO change this when systematics are implemented (also change naming)
os.environ["NUMUCC_SYST_DISK_ROOT"] = ""


In [4]:
from analysis_village.nueNp0Pi.event_selection import build_event_selection_pipeline
from pyanalib.event_selection_pipeline import EventSelectionPipelineConfig
from analysis_village.nueNp0Pi.config.datasets import PLOTS_BASE

today_str = datetime.now().strftime("%Y%m%d")
syst_tag = ""

# Optional overrides (None -> dated work dir under /exp/sbnd/data/users/$USER/...)
batch_work_base = f"/exp/sbnd/data/users/micarrig/nueNp0Pi/{today_str}"
batch_plots_dir = path.join(PLOTS_BASE, f"event_selection-{syst_tag}-{today_str}")
# batch_plots_dir = path.join(PLOTS_BASE, f"event_selection-debug")

batch_cfg = EventSelectionPipelineConfig(
    work_base=batch_work_base,
    plots_dir=batch_plots_dir,
    max_job_bytes=int(10.0 * 1024**3),  # 1 GiB per job
    mc_univ_syst=(), #("Flux", "G4", "GENIE"),
    skip_existing_batches=False,
    aggregate_only=False,
    skip_aggregate=False,
    save_fig=True,
    show_fig=False,
    # max_files_per_sample=1,  # smoke test: one file per sample
)
pipeline = build_event_selection_pipeline(batch_cfg)

records, jobs, manifest_path = pipeline.discover_jobs()
print(f"Batched workflow: {len(jobs)} job(s)  manifest={manifest_path}")
for j in jobs[:8]:
    print(f"  {j.sample} {j.tag}: {len(j.files)} file(s), {j.total_bytes / (1024**3):.3f} GiB")
if len(jobs) > 8:
    print(f"  ... and {len(jobs) - 8} more")


[event_selection] mc: looking in /exp/sbnd/data/users/micarrig/nueNp0Pi/selection_v2/*.df
[event_selection] mc: found 160 file(s)
[event_selection] sample=mc  files=160  total=399.61 GiB
[event_selection] 51 job(s) under size budget
  mc batch_0000: 4 file(s), 9.873 GiB
  mc batch_0001: 3 file(s), 7.549 GiB
  mc batch_0002: 3 file(s), 7.520 GiB
  mc batch_0003: 4 file(s), 9.045 GiB
  mc batch_0004: 4 file(s), 9.959 GiB
  mc batch_0005: 4 file(s), 9.948 GiB
  mc batch_0006: 4 file(s), 9.024 GiB
  mc batch_0007: 4 file(s), 9.461 GiB
  mc batch_0008: 4 file(s), 9.993 GiB
  mc batch_0009: 3 file(s), 7.584 GiB
  mc batch_0010: 3 file(s), 7.492 GiB
  mc batch_0011: 3 file(s), 7.604 GiB
  ... and 39 more
Batched workflow: 51 job(s)  manifest=/exp/sbnd/data/users/micarrig/nueNp0Pi/20260816/manifest.json
  mc batch_0000: 4 file(s), 9.873 GiB
  mc batch_0001: 3 file(s), 7.549 GiB
  mc batch_0002: 3 file(s), 7.520 GiB
  mc batch_0003: 4 file(s), 9.045 GiB
  mc batch_0004: 4 file(s), 9.959 GiB
  m

### Stacked event-distribution breakdown plots

By default, `build_pipeline()` (in `config/stages.py`) attaches a stacked breakdown plot
for every `EFFICIENCY_VARS` variable, by topology, at the final selection stage -- no
changes needed below to get them. Set the `EVT_BREAKDOWN_*` attributes on the
`config.stages` module before running the pipeline (cell below) to change the
breakdown type, which stage(s) they're attached to, or which variables get one.

In [5]:
batch_result = pipeline.run_full()

save_fig_dir = str(batch_result.plots_dir)
save_fig = batch_cfg.save_fig
show_plot = batch_cfg.show_fig
pot_str = batch_result.pot_str
data_tot_pot = batch_result.data_pot
merged_payload = batch_result.merged_payload

print("Batched workflow complete.")
print("  batches:", batch_result.batches_dir)
print("  plots  :", batch_result.plots_dir)
print("  manifest:", batch_result.manifest_path)
print("  POT    :", pot_str)

# Uncomment to preview PNGs inline (can be slow for many plots):
# pipeline.show_saved_plots(batch_result.plots_dir, max_images=20)

[event_selection] mc: looking in /exp/sbnd/data/users/micarrig/nueNp0Pi/selection_v2/*.df


[event_selection] mc: found 160 file(s)
[event_selection] sample=mc  files=160  total=399.61 GiB
[event_selection] 51 job(s) under size budget
  mc batch_0000: 4 file(s), 9.873 GiB
  mc batch_0001: 3 file(s), 7.549 GiB
  mc batch_0002: 3 file(s), 7.520 GiB
  mc batch_0003: 4 file(s), 9.045 GiB
  mc batch_0004: 4 file(s), 9.959 GiB
  mc batch_0005: 4 file(s), 9.948 GiB
  mc batch_0006: 4 file(s), 9.024 GiB
  mc batch_0007: 4 file(s), 9.461 GiB
  mc batch_0008: 4 file(s), 9.993 GiB
  mc batch_0009: 3 file(s), 7.584 GiB
  mc batch_0010: 3 file(s), 7.492 GiB
  mc batch_0011: 3 file(s), 7.604 GiB
  ... and 39 more
[event_selection] WORK_BASE=/exp/sbnd/data/users/micarrig/nueNp0Pi/20260816
[event_selection] BATCHES_DIR=/exp/sbnd/data/users/micarrig/nueNp0Pi/20260816/batches
[event_selection] sample=mc batch_0000  files=4  size=9.873 GiB


[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages: [pipeline] built 14 stages:
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [00] allreco                    cut=no     plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [01] is_fiducial                cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [02] is_flash_matched           cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [03] is_contained               cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [04] no_muons                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [05] no_pions                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [06] no_photons                 cut=ye

[event_selection] done sample=mc batch_0000  n_evt=2058043  pot=1.504e+19
[event_selection] sample=mc batch_0001  files=3  size=7.549 GiB


[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages: [pipeline] built 14 stages:
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [00] allreco                    cut=no     plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [01] is_fiducial                cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [02] is_flash_matched           cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [03] is_contained               cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [04] no_muons                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [05] no_pions                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [06] no_photons                 cut=ye

...

[aggregate] exposure totals: data_pot=0.000e+00 bnb_gates=0.000e+00 mc_pot=5.967e+20 dirt_pot=0.000e+00 intime_gates=0.000e+00 offbeam_gates=0.000e+00
[aggregate] applied global scales: {'scale_mc': 1.0, 'scale_dirt': 1.0, 'scale_intime': 0.0, 'scale_offbeam': 0.0}
[aggregate] data_pot (legend)=5.967e+20 -> POT label=5.97$\times 10^{20}$
[aggregate] systematics disk root: None


[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages: [pipeline] built 14 stages:
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [00] allreco                    cut=no     plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [01] is_fiducial                cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [02] is_flash_matched           cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [03] is_contained               cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [04] no_muons                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [05] no_pions                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [06] no_photons                 cut=ye

[aggregate] overlay plots without syst covariance (34 plots, 34 distinct var_save_name): E_nu, electron-dedx, electron-e, electron-e-res, electron-primary-score, electron-softmax-score, electron-vertex-distance, num-electrons, num-muons, num-photons, num-pions, num-protons, opening_angle, opening_angle-res, opening_angle_beam, opening_angle_beam-res, particle_ke, proton-p, proton-p-res, proton-softmax-score, secondary-proton-p, secondary-proton-p-res, tki-del_Tp, tki-del_Tp-res, tki-del_Tp_lp, tki-del_Tp_lp-res, tki-del_alpha, tki-del_alpha-res, tki-del_alpha_lp, tki-del_alpha_lp-res, tki-del_phi, tki-del_phi-res, tki-del_phi_lp, tki-del_phi_lp-res


[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages: [pipeline] built 14 stages:
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [00] allreco                    cut=no     plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [01] is_fiducial                cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [02] is_flash_matched           cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [03] is_contained               cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [04] no_muons                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [05] no_pions                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [06] no_photons                 cut=ye

[aggregate] final selection purity: 87.25% (weighted)  87.25% (raw counts, notebook-style)
[aggregate] wrote /nashome/m/micarrig/sbnd/nueCCNp/plots/event_selection--20260816/pkl/eff_dict.pkl
[aggregate] wrote /nashome/m/micarrig/sbnd/nueCCNp/plots/event_selection--20260816/pkl/merged_histdata.pkl
[event_selection] DONE plots -> /nashome/m/micarrig/sbnd/nueCCNp/plots/event_selection--20260816
Batched workflow complete.
  batches: /exp/sbnd/data/users/micarrig/nueNp0Pi/20260816/batches
  plots  : /nashome/m/micarrig/sbnd/nueCCNp/plots/event_selection--20260816
  manifest: /exp/sbnd/data/users/micarrig/nueNp0Pi/20260816/manifest.json
  POT    : 5.97$\times 10^{20}$


In [6]:
import analysis_village.nueNp0Pi.config.stages as stages_mod

# Uncomment/edit any of these before running the pipeline below. Defaults shown.
stages_mod.EVT_BREAKDOWN_TYPE = "genie"           # "topology" | "genie" | "pdg"
stages_mod.EVT_BREAKDOWN_DIR_NAME = f"selection_{stages_mod.EVT_BREAKDOWN_TYPE}"
# stages_mod.EVT_BREAKDOWN_STAGE_KEYS = None            # None -> final stage only; or e.g. ["no_photons", "electron_dedx"]
# stages_mod.EVT_BREAKDOWN_VARS = stages_mod.EFFICIENCY_VARS  # or e.g. [VariableConfig.neutrino_energy()]
# stages_mod.EVT_BREAKDOWN_VARS = []                    # uncomment to disable these plots entirely

# Output subdirectory for these plots (under the plots dir), default "selection". Set this
# alongside EVT_BREAKDOWN_TYPE if you want to run more than one breakdown_type (e.g. once as
# "topology", once as "genie") without the later run's selection_<var>.png files
# overwriting the earlier run's -- e.g.:
# stages_mod.EVT_BREAKDOWN_DIR_NAME = f"selection_{stages_mod.EVT_BREAKDOWN_TYPE}"


In [7]:
batch_result = pipeline.run_full()

save_fig_dir = str(batch_result.plots_dir)
save_fig = batch_cfg.save_fig
show_plot = batch_cfg.show_fig
pot_str = batch_result.pot_str
data_tot_pot = batch_result.data_pot
merged_payload = batch_result.merged_payload

print("Batched workflow complete.")
print("  batches:", batch_result.batches_dir)
print("  plots  :", batch_result.plots_dir)
print("  manifest:", batch_result.manifest_path)
print("  POT    :", pot_str)

# Uncomment to preview PNGs inline (can be slow for many plots):
# pipeline.show_saved_plots(batch_result.plots_dir, max_images=20)


[event_selection] mc: looking in /exp/sbnd/data/users/micarrig/nueNp0Pi/selection_v2/*.df
[event_selection] mc: found 160 file(s)
[event_selection] sample=mc  files=160  total=399.61 GiB
[event_selection] 51 job(s) under size budget
  mc batch_0000: 4 file(s), 9.873 GiB
  mc batch_0001: 3 file(s), 7.549 GiB
  mc batch_0002: 3 file(s), 7.520 GiB
  mc batch_0003: 4 file(s), 9.045 GiB
  mc batch_0004: 4 file(s), 9.959 GiB
  mc batch_0005: 4 file(s), 9.948 GiB
  mc batch_0006: 4 file(s), 9.024 GiB
  mc batch_0007: 4 file(s), 9.461 GiB
  mc batch_0008: 4 file(s), 9.993 GiB
  mc batch_0009: 3 file(s), 7.584 GiB
  mc batch_0010: 3 file(s), 7.492 GiB
  mc batch_0011: 3 file(s), 7.604 GiB
  ... and 39 more
[event_selection] WORK_BASE=/exp/sbnd/data/users/micarrig/nueNp0Pi/20260816
[event_selection] BATCHES_DIR=/exp/sbnd/data/users/micarrig/nueNp0Pi/20260816/batches
[event_selection] sample=mc batch_0000  files=4  size=9.873 GiB


[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages: [pipeline] built 14 stages:
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [00] allreco                    cut=no     plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [01] is_fiducial                cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [02] is_flash_matched           cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [03] is_contained               cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [04] no_muons                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [05] no_pions                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [06] no_photons                 cut=ye

[DEBUG] cafpyana.pyanalib.chunked_selection: [pipeline]     efficiency accumulators …
[DEBUG] cafpyana.pyanalib.chunked_selection: [pipeline] <<< stage='allreco' finished
[DEBUG] cafpyana.pyanalib.chunked_selection: [pipeline] >>> stage='is_fiducial' plots=0 breakdown=True eff=True
[DEBUG] cafpyana.pyanalib.chunked_selection: [pipeline]     executing cut '_cut' …
[DEBUG] cafpyana.pyanalib.chunked_selection: [pipeline]     after cut: len(evt)=138179 len(trk)=1490936
[DEBUG] cafpyana.pyanalib.chunked_selection: [pipeline]     bar breakdown …
[DEBUG] cafpyana.pyanalib.chunked_selection: [pipeline]     efficiency accumulators …
[DEBUG] cafpyana.pyanalib.chunked_selection: [pipeline] <<< stage='is_fiducial' finished
[DEBUG] cafpyana.pyanalib.chunked_selection: [pipeline] >>> stage='is_flash_matched' plots=0 breakdown=True eff=True
[DEBUG] cafpyana.pyanalib.chunked_selection: [pipeline]     executing cut '_cut' …
[DEBUG] cafpyana.pyanalib.chunked_selection: [pipeline]     after cut: len(evt)

[event_selection] done sample=mc batch_0000  n_evt=2058043  pot=1.504e+19
[event_selection] sample=mc batch_0001  files=3  size=7.549 GiB


[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages: [pipeline] built 14 stages:
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [00] allreco                    cut=no     plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [01] is_fiducial                cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [02] is_flash_matched           cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [03] is_contained               cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [04] no_muons                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [05] no_pions                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [06] no_photons                 cut=ye

...

[aggregate] exposure totals: data_pot=0.000e+00 bnb_gates=0.000e+00 mc_pot=5.967e+20 dirt_pot=0.000e+00 intime_gates=0.000e+00 offbeam_gates=0.000e+00
[aggregate] applied global scales: {'scale_mc': 1.0, 'scale_dirt': 1.0, 'scale_intime': 0.0, 'scale_offbeam': 0.0}
[aggregate] data_pot (legend)=5.967e+20 -> POT label=5.97$\times 10^{20}$
[aggregate] systematics disk root: None


[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages: [pipeline] built 14 stages:
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [00] allreco                    cut=no     plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [01] is_fiducial                cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [02] is_flash_matched           cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [03] is_contained               cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [04] no_muons                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [05] no_pions                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [06] no_photons                 cut=ye

[aggregate] WARN: ratio_mode='signal_bkgd' on 'proton-p-res' expects breakdown_type='topology' but this plot's breakdown_type is 'genie' -- skipping ratio panel (the category indices aren't valid for a different breakdown_type's category order/count).
[aggregate] WARN: ratio_mode='signal_bkgd' on 'tki-del_phi-res' expects breakdown_type='topology' but this plot's breakdown_type is 'genie' -- skipping ratio panel (the category indices aren't valid for a different breakdown_type's category order/count).
[aggregate] WARN: ratio_mode='signal_bkgd' on 'tki-del_alpha-res' expects breakdown_type='topology' but this plot's breakdown_type is 'genie' -- skipping ratio panel (the category indices aren't valid for a different breakdown_type's category order/count).
[aggregate] WARN: ratio_mode='signal_bkgd' on 'tki-del_alpha_lp-res' expects breakdown_type='topology' but this plot's breakdown_type is 'genie' -- skipping ratio panel (the category indices aren't valid for a different breakdown_type's

[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages: [pipeline] built 14 stages:
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [00] allreco                    cut=no     plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [01] is_fiducial                cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [02] is_flash_matched           cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [03] is_contained               cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [04] no_muons                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [05] no_pions                   cut=yes    plots=0 (—)  flags=eff,breakdown
[DEBUG] cafpyana.analysis_village.nueNp0Pi.config.stages:   [06] no_photons                 cut=ye

[aggregate] final selection purity: 87.25% (weighted)  87.25% (raw counts, notebook-style)
[aggregate] wrote /nashome/m/micarrig/sbnd/nueCCNp/plots/event_selection--20260816/pkl/eff_dict.pkl
[aggregate] wrote /nashome/m/micarrig/sbnd/nueCCNp/plots/event_selection--20260816/pkl/merged_histdata.pkl
[event_selection] DONE plots -> /nashome/m/micarrig/sbnd/nueCCNp/plots/event_selection--20260816
Batched workflow complete.
  batches: /exp/sbnd/data/users/micarrig/nueNp0Pi/20260816/batches
  plots  : /nashome/m/micarrig/sbnd/nueCCNp/plots/event_selection--20260816
  manifest: /exp/sbnd/data/users/micarrig/nueNp0Pi/20260816/manifest.json
  POT    : 5.97$\times 10^{20}$
